# Agent Framework Benchmark — Results Analysis

This notebook loads benchmark results and creates interactive Plotly charts for comparison.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Output directory for figures
FIGURES_DIR = Path("../results/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load results
results_path = Path("../results/benchmark_results.csv")
if results_path.exists():
    df = pd.read_csv(results_path)
    print(f"Loaded {len(df)} results")
    display(df.head())
else:
    df = None
    print(f"No results found at {results_path}")
    print("Run the benchmark first: uv run python -m benchmark.runner")
    print("Subsequent cells will be skipped.")

Loaded 45 results


,framework,company,iteration,completeness,accuracy,structure,insight,readability,overall,reasoning,latency_seconds,input_tokens,output_tokens,total_tokens,estimated_cost,report_text
0,langgraph,Anthropic,1,9.0,10.0,10.0,10.0,10.0,9.8,"The report is exceptionally well-structured, a...",494.54,3487,5230,8717,0.0,**Research Report: Anthropic** \n\n---\n\n**E...
1,langgraph,Anthropic,2,9.0,9.0,10.0,9.0,9.0,9.2,"The report is highly comprehensive, covering a...",469.60,3487,5230,8717,0.0,**Research Report: Anthropic** \n\n---\n\n**E...
2,langgraph,Anthropic,3,9.0,8.0,10.0,9.0,10.0,9.2,"The report is comprehensive, well-structured, ...",468.82,3487,5230,8717,0.0,**Research Report: Anthropic** \n\n---\n\n**E...
3,langgraph,Stripe,1,9.0,10.0,10.0,9.0,10.0,9.4,"The report is comprehensive, well-structured, ...",388.04,3420,5385,8805,0.0,**Research Report: Stripe** \n\n---\n\n### **...
4,langgraph,Stripe,2,9.0,9.0,10.0,9.0,10.0,9.2,"The report is comprehensive, covering key aspe...",527.10,3420,5385,8805,0.0,**Research Report: Stripe** \n\n---\n\n### **...


## Quality Scores by Framework

In [ ]:
if df is not None:
    # Grouped bar chart: quality scores per framework
    quality_cols = ["completeness", "accuracy", "structure", "insight", "readability", "overall"]
    quality_avg = df.groupby("framework")[quality_cols].mean().reset_index()
    quality_melted = quality_avg.melt(id_vars="framework", var_name="metric", value_name="score")

    fig = px.bar(
        quality_melted,
        x="framework",
        y="score",
        color="metric",
        barmode="group",
        title="Average Quality Scores by Framework",
        labels={"score": "Score", "framework": "Framework"},
    )
    fig.update_layout(yaxis_range=[8, 10.2])
    fig.show()
    fig.write_image(FIGURES_DIR / "01_quality_scores.png", width=1200, height=600, scale=2)

## Latency Comparison

In [3]:
if df is not None:
    # Bar chart: average latency per framework
    latency_avg = df.groupby("framework")["latency_seconds"].mean().reset_index()
    latency_avg = latency_avg.sort_values("latency_seconds")

    fig = px.bar(
        latency_avg,
        x="framework",
        y="latency_seconds",
        title="Average Latency by Framework",
        labels={"latency_seconds": "Latency (seconds)", "framework": "Framework"},
        color="framework",
    )
    fig.show()
    fig.write_image(FIGURES_DIR / "02_latency.png", width=1200, height=600, scale=2)

## Token Usage

In [4]:
if df is not None:
    # Bar chart: average token usage per framework
    token_avg = df.groupby("framework")[["input_tokens", "output_tokens"]].mean().reset_index()
    token_melted = token_avg.melt(id_vars="framework", var_name="type", value_name="tokens")

    fig = px.bar(
        token_melted,
        x="framework",
        y="tokens",
        color="type",
        barmode="stack",
        title="Average Token Usage by Framework",
        labels={"tokens": "Tokens", "framework": "Framework"},
    )
    fig.show()
    fig.write_image(FIGURES_DIR / "03_token_usage.png", width=1200, height=600, scale=2)

## Cost Comparison

In [5]:
if df is not None:
    # Table: cost comparison
    cost_summary = df.groupby("framework").agg({
        "estimated_cost": ["mean", "sum"],
        "total_tokens": "mean",
        "latency_seconds": "mean",
        "overall": "mean",
    }).round(4)

    cost_summary.columns = ["Avg Cost ($)", "Total Cost ($)", "Avg Tokens", "Avg Latency (s)", "Avg Quality"]
    display(cost_summary)

,Avg Cost ($),Total Cost ($),Avg Tokens,Avg Latency (s),Avg Quality
framework,,,,,
agents_sdk,0.0,0.0,8676.3333,448.3322,9.3111
autogen,0.0,0.0,10793.3333,571.7400,9.6333
crewai,0.0,0.0,27684.0000,245.8044,9.6556
langgraph,0.0,0.0,8822.5556,506.1344,9.4222
msagent,0.0,0.0,0.0000,92.9211,9.8667


## Score Distribution (Box Plots)

In [ ]:
if df is not None:
    # Box plots: score distribution per framework
    fig = px.box(
        df,
        x="framework",
        y="overall",
        color="framework",
        title="Overall Quality Score Distribution by Framework",
        labels={"overall": "Overall Score", "framework": "Framework"},
        points="all",
    )
    fig.update_layout(yaxis_range=[8, 10.2])
    fig.show()
    fig.write_image(FIGURES_DIR / "04_score_distribution.png", width=1200, height=600, scale=2)

In [ ]:
if df is not None:
    # Box plots for all quality metrics
    quality_df = df.melt(
        id_vars=["framework"],
        value_vars=quality_cols,
        var_name="metric",
        value_name="score",
    )

    fig = px.box(
        quality_df,
        x="metric",
        y="score",
        color="framework",
        title="Quality Metric Distributions by Framework",
        labels={"score": "Score", "metric": "Metric"},
    )
    fig.update_layout(yaxis_range=[7, 10.3])
    fig.show()
    fig.write_image(FIGURES_DIR / "05_quality_metric_distributions.png", width=1200, height=600, scale=2)

## Summary Table

In [8]:
if df is not None:
    # Comprehensive summary table
    summary = df.groupby("framework").agg({
        "overall": "mean",
        "completeness": "mean",
        "accuracy": "mean",
        "structure": "mean",
        "insight": "mean",
        "readability": "mean",
        "latency_seconds": "mean",
        "total_tokens": "mean",
        "estimated_cost": "sum",
    }).round(2)

    summary.columns = [
        "Quality", "Completeness", "Accuracy", "Structure",
        "Insight", "Readability", "Latency (s)", "Tokens", "Total Cost ($)",
    ]

    # Rank by overall quality
    summary = summary.sort_values("Quality", ascending=False)
    display(summary)

,Quality,Completeness,Accuracy,Structure,Insight,Readability,Latency (s),Tokens,Total Cost ($)
framework,,,,,,,,,
msagent,9.87,10.00,10.00,10.00,9.33,10.00,92.92,0.00,0.0
crewai,9.66,9.44,9.44,9.89,9.56,10.00,245.80,27684.00,0.0
autogen,9.63,9.44,9.67,9.89,9.33,9.89,571.74,10793.33,0.0
langgraph,9.42,9.11,9.44,9.89,9.22,9.78,506.13,8822.56,0.0
agents_sdk,9.31,9.00,9.11,9.89,9.00,9.78,448.33,8676.33,0.0


## Per-Company Quality Breakdown

In [ ]:
if df is not None:
    # Faceted bar chart: quality per company per framework
    company_quality = df.groupby(["framework", "company"])["overall"].agg(["mean", "std"]).reset_index()
    company_quality.columns = ["framework", "company", "mean_quality", "std_quality"]

    fig = px.bar(
        company_quality,
        x="company",
        y="mean_quality",
        color="framework",
        barmode="group",
        error_y="std_quality",
        title="Average Quality Score by Company and Framework",
        labels={"mean_quality": "Overall Score", "company": "Company"},
    )
    fig.update_layout(yaxis_range=[8, 10.5])
    fig.show()
    fig.write_image(FIGURES_DIR / "06_per_company_quality.png", width=1200, height=600, scale=2)

## Consistency Analysis (Standard Deviation Across Iterations)

In [10]:
if df is not None:
    # Consistency: lower std = more consistent results
    consistency = df.groupby(["framework", "company"]).agg({
        "overall": ["mean", "std"],
        "latency_seconds": ["mean", "std"],
    }).round(2)
    consistency.columns = ["Avg Quality", "Quality Std", "Avg Latency", "Latency Std"]
    consistency = consistency.reset_index()

    print("Consistency across iterations (lower std = more consistent):")
    display(consistency)

    # Framework-level consistency
    fw_consistency = df.groupby("framework")["overall"].agg(["mean", "std", "min", "max"]).round(2)
    fw_consistency.columns = ["Mean", "Std Dev", "Min", "Max"]
    fw_consistency["Range"] = fw_consistency["Max"] - fw_consistency["Min"]
    print("\nFramework-level quality consistency:")
    display(fw_consistency.sort_values("Std Dev"))

Consistency across iterations (lower std = more consistent):


,framework,company,Avg Quality,Quality Std,Avg Latency,Latency Std
0,agents_sdk,Anthropic,8.97,0.47,554.81,130.42
1,agents_sdk,Datadog,9.50,0.10,301.15,105.36
2,agents_sdk,Stripe,9.47,0.06,489.04,100.04
3,autogen,Anthropic,9.20,0.53,544.70,76.54
4,autogen,Datadog,9.77,0.25,550.83,12.67
5,autogen,Stripe,9.93,0.12,619.69,157.51
6,crewai,Anthropic,9.53,0.31,267.33,3.04
7,crewai,Datadog,9.70,0.36,251.20,12.95
8,crewai,Stripe,9.73,0.31,218.89,3.39
9,langgraph,Anthropic,9.40,0.35,477.65,14.63



Framework-level quality consistency:


,Mean,Std Dev,Min,Max,Range
framework,,,,,
msagent,9.87,0.10,9.8,10.0,0.2
crewai,9.66,0.30,9.2,10.0,0.8
langgraph,9.42,0.32,9.0,10.0,1.0
agents_sdk,9.31,0.36,8.6,9.6,1.0
autogen,9.63,0.45,8.6,10.0,1.4


## Quality vs Latency Trade-off

In [ ]:
if df is not None:
    # Scatter plot: quality vs latency (each dot = one run)
    # Use fixed marker size since token counts vary too much across frameworks
    fig = px.scatter(
        df,
        x="latency_seconds",
        y="overall",
        color="framework",
        symbol="company",
        title="Quality vs Latency Trade-off",
        labels={
            "latency_seconds": "Latency (seconds)",
            "overall": "Overall Quality Score",
        },
        hover_data=["company", "iteration", "total_tokens"],
    )
    fig.update_traces(marker=dict(size=10))
    fig.update_layout(yaxis_range=[8, 10.2])
    fig.show()
    fig.write_image(FIGURES_DIR / "07_quality_vs_latency.png", width=1200, height=600, scale=2)

## Radar Chart — Framework Quality Profiles

In [ ]:
if df is not None:
    # Radar chart comparing quality dimensions per framework
    radar_cols = ["completeness", "accuracy", "structure", "insight", "readability"]
    radar_avg = df.groupby("framework")[radar_cols].mean()

    fig = go.Figure()
    for fw in radar_avg.index:
        values = radar_avg.loc[fw].tolist()
        values.append(values[0])  # close the polygon
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=radar_cols + [radar_cols[0]],
            fill="toself",
            name=fw,
            opacity=0.6,
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[8.5, 10.2])),
        title="Quality Dimension Profiles by Framework",
        showlegend=True,
    )
    fig.show()
    fig.write_image(FIGURES_DIR / "08_radar_quality_profiles.png", width=1200, height=700, scale=2)

## Report Length Analysis

In [ ]:
if df is not None:
    # Report length: word count and character count
    df["report_chars"] = df["report_text"].str.len()
    df["report_words"] = df["report_text"].str.split().str.len()

    fig = px.box(
        df,
        x="framework",
        y="report_words",
        color="framework",
        title="Report Length (Word Count) by Framework",
        labels={"report_words": "Word Count", "framework": "Framework"},
        points="all",
    )
    fig.show()
    fig.write_image(FIGURES_DIR / "09_report_length.png", width=1200, height=600, scale=2)

    # Report length vs quality correlation
    fig2 = px.scatter(
        df,
        x="report_words",
        y="overall",
        color="framework",
        title="Report Length vs Quality Score",
        labels={"report_words": "Word Count", "overall": "Overall Quality"},
        hover_data=["company", "iteration"],
        trendline="ols",
    )
    fig2.update_layout(yaxis_range=[8, 10.2])
    fig2.show()
    fig2.write_image(FIGURES_DIR / "10_report_length_vs_quality.png", width=1200, height=600, scale=2)

    print(f"\nReport length statistics:")
    display(df.groupby("framework")[["report_words", "report_chars"]].describe().round(0))

## Statistical Significance Testing

Are the quality differences between frameworks statistically significant, or just noise?
We use the Kruskal-Wallis H-test (non-parametric, doesn't assume normality) followed by
pairwise Mann-Whitney U tests with Bonferroni correction.

In [14]:
if df is not None:
    from scipy import stats
    from itertools import combinations

    frameworks = df["framework"].unique()
    groups = [df[df["framework"] == fw]["overall"].values for fw in frameworks]

    # Step 1: Kruskal-Wallis H-test (are ANY groups different?)
    h_stat, kw_p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis H-test: H={h_stat:.3f}, p={kw_p:.4f}")
    if kw_p < 0.05:
        print("  -> Significant difference exists between at least two frameworks (p < 0.05)")
    else:
        print("  -> No significant difference found between frameworks (p >= 0.05)")

    # Step 2: Pairwise Mann-Whitney U tests with Bonferroni correction
    n_comparisons = len(list(combinations(frameworks, 2)))
    alpha_corrected = 0.05 / n_comparisons

    print(f"\nPairwise Mann-Whitney U tests (Bonferroni-corrected alpha = {alpha_corrected:.4f}):")
    print(f"{'Pair':<35} {'U-stat':>8} {'p-value':>10} {'Significant?':>14} {'Effect size r':>14}")
    print("-" * 85)

    for fw_a, fw_b in combinations(frameworks, 2):
        a = df[df["framework"] == fw_a]["overall"].values
        b = df[df["framework"] == fw_b]["overall"].values
        u_stat, p_val = stats.mannwhitneyu(a, b, alternative="two-sided")
        # Effect size r = Z / sqrt(N)
        z_stat = stats.norm.ppf(1 - p_val / 2) if p_val < 1.0 else 0
        r_effect = z_stat / (len(a) + len(b)) ** 0.5
        sig = "YES" if p_val < alpha_corrected else "no"
        print(f"  {fw_a} vs {fw_b:<20} {u_stat:>8.1f} {p_val:>10.4f} {sig:>14} {r_effect:>14.3f}")

    print(f"\nEffect size interpretation: r < 0.3 = small, 0.3-0.5 = medium, > 0.5 = large")

    # Step 3: Also test latency differences
    print(f"\n--- Latency significance ---")
    lat_groups = [df[df["framework"] == fw]["latency_seconds"].values for fw in frameworks]
    h_lat, kw_lat_p = stats.kruskal(*lat_groups)
    print(f"Kruskal-Wallis H-test (latency): H={h_lat:.3f}, p={kw_lat_p:.6f}")
    if kw_lat_p < 0.05:
        print("  -> Significant latency difference exists between frameworks")

Kruskal-Wallis H-test: H=14.751, p=0.0052
  -> Significant difference exists between at least two frameworks (p < 0.05)

Pairwise Mann-Whitney U tests (Bonferroni-corrected alpha = 0.0050):
Pair                                  U-stat    p-value   Significant?  Effect size r
-------------------------------------------------------------------------------------
  langgraph vs autogen                  23.0     0.1287             no          0.358
  langgraph vs agents_sdk               41.0     1.0000             no          0.000
  langgraph vs crewai                   23.5     0.1397             no          0.348
  langgraph vs msagent                  10.5     0.0068             no          0.638
  autogen vs agents_sdk               63.0     0.0483             no          0.465
  autogen vs crewai                   44.0     0.7865             no          0.064
  autogen vs msagent                  28.5     0.2783             no          0.256
  agents_sdk vs crewai                   2

## Token Efficiency

Quality-per-token measures how much value each framework extracts from its LLM calls.
Higher is better — achieving the same quality with fewer tokens means lower cost in production.

In [ ]:
if df is not None:
    # Only compute for frameworks that actually tracked tokens
    df_with_tokens = df[df["total_tokens"] > 0].copy()

    if len(df_with_tokens) > 0:
        # Quality per 1K tokens
        df_with_tokens["quality_per_1k_tokens"] = (
            df_with_tokens["overall"] / df_with_tokens["total_tokens"] * 1000
        )
        # Quality per second of latency
        df_with_tokens["quality_per_second"] = (
            df_with_tokens["overall"] / df_with_tokens["latency_seconds"]
        )

        efficiency = df_with_tokens.groupby("framework").agg({
            "quality_per_1k_tokens": "mean",
            "quality_per_second": "mean",
            "total_tokens": "mean",
            "overall": "mean",
            "latency_seconds": "mean",
        }).round(4)
        efficiency.columns = [
            "Quality / 1K Tokens", "Quality / Second",
            "Avg Tokens", "Avg Quality", "Avg Latency (s)",
        ]
        efficiency = efficiency.sort_values("Quality / 1K Tokens", ascending=False)

        print("Token Efficiency (higher = better value per token):")
        display(efficiency)

        # Use mean per framework instead of individual run bars
        fig = px.bar(
            efficiency.reset_index(),
            x="framework",
            y="Quality / 1K Tokens",
            color="framework",
            title="Token Efficiency: Quality Score per 1K Tokens (higher = better)",
            labels={"Quality / 1K Tokens": "Quality / 1K Tokens", "framework": "Framework"},
        )
        fig.show()
        fig.write_image(FIGURES_DIR / "11_token_efficiency.png", width=1200, height=600, scale=2)
    else:
        print("No frameworks with token tracking data available.")
        print("Token efficiency analysis requires total_tokens > 0.")

## Latency Breakdown Heatmap

Shows average latency for each framework-company pair.
Reveals whether certain research targets are inherently harder (more data to process,
more complex to analyze) regardless of framework choice.

In [16]:
if df is not None:
    import numpy as np

    # Pivot: framework (rows) x company (columns), values = avg latency
    latency_pivot = df.pivot_table(
        values="latency_seconds",
        index="framework",
        columns="company",
        aggfunc="mean",
    ).round(1)

    # Also create a quality pivot for a side-by-side heatmap
    quality_pivot = df.pivot_table(
        values="overall",
        index="framework",
        columns="company",
        aggfunc="mean",
    ).round(2)

    # Latency heatmap
    fig = px.imshow(
        latency_pivot,
        text_auto=True,
        color_continuous_scale="YlOrRd",
        title="Average Latency (seconds) — Framework x Company",
        labels=dict(x="Company", y="Framework", color="Latency (s)"),
        aspect="auto",
    )
    fig.show()
    fig.write_image(FIGURES_DIR / "12_latency_heatmap.png", width=1200, height=600, scale=2)

    # Quality heatmap
    fig2 = px.imshow(
        quality_pivot,
        text_auto=True,
        color_continuous_scale="RdYlGn",
        title="Average Quality Score — Framework x Company",
        labels=dict(x="Company", y="Framework", color="Quality"),
        aspect="auto",
        zmin=7, zmax=10,
    )
    fig2.show()
    fig2.write_image(FIGURES_DIR / "13_quality_heatmap.png", width=1200, height=600, scale=2)

    # Which company is hardest/easiest across all frameworks?
    print("Company difficulty (avg latency across all frameworks):")
    company_difficulty = df.groupby("company").agg({
        "latency_seconds": "mean",
        "overall": "mean",
    }).round(1)
    company_difficulty.columns = ["Avg Latency (s)", "Avg Quality"]
    display(company_difficulty.sort_values("Avg Latency (s)", ascending=False))

Company difficulty (avg latency across all frameworks):


,Avg Latency (s),Avg Quality
company,,
Anthropic,388.0,9.4
Stripe,379.3,9.7
Datadog,351.7,9.6
